# Titanic modeling

Continue from the same cleaned offline file written by `01_eda.ipynb` / `01_eda.py`. This notebook never calls `sns.load_dataset`. Every preprocessing step is fit on the training split only.



In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    auc,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree

%matplotlib inline

ROOT = Path.cwd()
CSV_PATH = ROOT / "titanic.csv"
FIG = ROOT / "figures"
REPORT_PATH = ROOT / "modeling_report.json"
PIPELINE_PATH = ROOT / "survival_pipeline.joblib"
FIG.mkdir(exist_ok=True)

FEATURES = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
NUMERIC = ["age", "fare", "sibsp", "parch", "pclass"]
CATEGORICAL = ["sex", "embarked"]
TARGET = "survived"

if not CSV_PATH.exists():
    raise SystemExit("titanic.csv is missing. Run 01_eda.ipynb or 01_eda.py first.")

frame = pd.read_csv(CSV_PATH)
frame.head()



## Stratified train/test split

Survival is the minority class (~38%). Stratifying keeps that rate stable across folds so accuracy is comparable.



In [ ]:
x = frame[FEATURES]
y = frame[TARGET]
balance = {
    "not_survived": int((y == 0).sum()),
    "survived": int((y == 1).sum()),
    "survived_share": round(float(y.mean()), 4),
}
print("class_balance", balance)

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)
print("train_size", len(x_train), "test_size", len(x_test))
print("train_survived_share", round(float(y_train.mean()), 4))
print("test_survived_share", round(float(y_test.mean()), 4))



## Preprocessing (fit on train only)

Numeric columns: median impute + `StandardScaler`. Categorical (`sex`, `embarked`): most-frequent impute + one-hot. Wrapped in a `ColumnTransformer` inside a `Pipeline` so the test split is transform-only.



In [ ]:
def build_preprocessor() -> ColumnTransformer:
    numeric = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("num", numeric, NUMERIC),
            ("cat", categorical, CATEGORICAL),
        ]
    )


def classification_metrics(y_true, y_pred, y_prob) -> dict:
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    return {
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
        "accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
        "precision": round(float(precision_score(y_true, y_pred, zero_division=0)), 4),
        "recall": round(float(recall_score(y_true, y_pred, zero_division=0)), 4),
        "f1": round(float(f1_score(y_true, y_pred, zero_division=0)), 4),
        "auc": round(float(auc(fpr, tpr)), 4),
        "fpr": [round(float(v), 4) for v in fpr],
        "tpr": [round(float(v), 4) for v in tpr],
    }


def fit_classifier(name, estimator, x_train, y_train, x_test, y_test):
    pipeline = Pipeline(
        steps=[
            ("preprocess", build_preprocessor()),
            ("model", estimator),
        ]
    )
    pipeline.fit(x_train, y_train)
    pred = pipeline.predict(x_test)
    prob = pipeline.predict_proba(x_test)[:, 1]
    metrics = classification_metrics(y_test, pred, prob)
    metrics["model"] = name
    return pipeline, metrics



## Train three classifiers



In [ ]:
specs = [
    ("logistic_regression", LogisticRegression(max_iter=500)),
    ("decision_tree", DecisionTreeClassifier(random_state=42)),
    ("random_forest", RandomForestClassifier(random_state=42)),
]

fitted = {}
metrics = []
for name, estimator in specs:
    pipeline, scores = fit_classifier(name, estimator, x_train, y_train, x_test, y_test)
    fitted[name] = pipeline
    metrics.append(scores)
    print(name, {k: scores[k] for k in ("accuracy", "precision", "recall", "f1", "auc")})

comparison = pd.DataFrame(
    [{k: item[k] for k in ("model", "accuracy", "precision", "recall", "f1", "auc")} for item in metrics]
)
comparison



In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for item in metrics:
    ax.plot(item["fpr"], item["tpr"], label=f"{item['model']} (AUC={item['auc']:.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("ROC curves")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "roc_curves.png", dpi=120)
plt.show()

tree_pipeline = fitted["decision_tree"]
feature_names = list(tree_pipeline.named_steps["preprocess"].get_feature_names_out())
fig, ax = plt.subplots(figsize=(22, 10))
plot_tree(
    tree_pipeline.named_steps["model"],
    feature_names=feature_names,
    class_names=["not_survived", "survived"],
    filled=True,
    max_depth=3,
    fontsize=8,
    ax=ax,
)
ax.set_title("Decision tree (top 3 levels of the fitted tree)")
fig.tight_layout()
fig.savefig(FIG / "decision_tree.png", dpi=120)
plt.show()



## Imbalance handling

Retrain logistic regression three ways. SMOTE is applied to the **training fold only** after preprocessing.



In [ ]:
imbalance = []
_, m0 = fit_classifier("logreg_baseline", LogisticRegression(max_iter=500), x_train, y_train, x_test, y_test)
imbalance.append({k: m0[k] for k in ("model", "precision", "recall", "f1")})

_, m1 = fit_classifier(
    "logreg_class_weight",
    LogisticRegression(max_iter=500, class_weight="balanced"),
    x_train, y_train, x_test, y_test,
)
imbalance.append({k: m1[k] for k in ("model", "precision", "recall", "f1")})

preprocessor = build_preprocessor()
x_train_ready = preprocessor.fit_transform(x_train)
x_test_ready = preprocessor.transform(x_test)
x_resampled, y_resampled = SMOTE(random_state=42).fit_resample(x_train_ready, y_train)
smote_model = LogisticRegression(max_iter=500)
smote_model.fit(x_resampled, y_resampled)
pred = smote_model.predict(x_test_ready)
prob = smote_model.predict_proba(x_test_ready)[:, 1]
m2 = classification_metrics(y_test, pred, prob)
imbalance.append(
    {
        "model": "logreg_smote_train_only",
        "precision": m2["precision"],
        "recall": m2["recall"],
        "f1": m2["f1"],
        "train_rows_before": int(len(y_train)),
        "train_rows_after_smote": int(len(y_resampled)),
    }
)
pd.DataFrame(imbalance)



SMOTE raises recall and F1 the most while keeping precision a bit higher than `class_weight='balanced'`. For a survival decision, missing a survivor is costlier than a false positive, so the F1 gain is the better trade.



## Random forest hyperparameter tuning + OOB score



In [ ]:
search_pipeline = Pipeline(
    steps=[
        ("preprocess", build_preprocessor()),
        ("model", RandomForestClassifier(oob_score=True, random_state=42)),
    ]
)
search = GridSearchCV(
    search_pipeline,
    param_grid={
        "model__n_estimators": [100, 200],
        "model__max_depth": [4, 8, None],
        "model__max_features": ["sqrt", "log2"],
    },
    cv=3,
    scoring="f1",
    n_jobs=1,
)
search.fit(x_train, y_train)
best = search.best_estimator_.named_steps["model"]
tuning = {
    "best_params": search.best_params_,
    "best_cv_f1": round(float(search.best_score_), 4),
    "oob_score": round(float(best.oob_score_), 4),
}
tuning



## Regression side-task: predict fare

Multivariate linear regression on the same dataset. Classification and regression metrics stay in separate groups.



In [ ]:
predictors = ["pclass", "sex", "age", "sibsp", "parch", "embarked", "survived"]
work = frame[predictors + ["fare"]].copy()
xr_train, xr_test, yr_train, yr_test = train_test_split(
    work[predictors], work["fare"], test_size=0.2, random_state=42
)

reg_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            ["age", "sibsp", "parch", "pclass", "survived"],
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]),
            ["sex", "embarked"],
        ),
    ]
)
reg_model = Pipeline([("preprocess", reg_preprocessor), ("model", LinearRegression())])
reg_model.fit(xr_train, yr_train)
pred = reg_model.predict(xr_test)
residual = yr_test.to_numpy() - pred
r2 = float(r2_score(yr_test, pred))
n = len(yr_test)
p = reg_model.named_steps["preprocess"].get_feature_names_out().shape[0]
adjusted = 1 - (1 - r2) * (n - 1) / (n - p - 1)
spread_corr = float(np.corrcoef(np.abs(residual), pred)[0, 1])

regression = {
    "mae": round(float(mean_absolute_error(yr_test, pred)), 4),
    "rmse": round(float(np.sqrt(mean_squared_error(yr_test, pred))), 4),
    "r2": round(r2, 4),
    "adjusted_r2": round(float(adjusted), 4),
    "abs_residual_vs_fitted_corr": round(spread_corr, 4),
    "heteroscedasticity": "present" if spread_corr > 0.2 else "not clearly present",
}
print(regression)

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(pred, residual, alpha=0.7)
ax.axhline(0, color="gray", linestyle="--")
ax.set_xlabel("Fitted fare")
ax.set_ylabel("Residual")
ax.set_title("Fare regression residuals")
fig.tight_layout()
fig.savefig(FIG / "fare_residuals.png", dpi=120)
plt.show()



Residuals fan out as fitted fare grows, so heteroscedasticity is present. That matches the right-skewed fare distribution from EDA.



## Final comparison, recommendation, and saved pipeline



In [ ]:
classifier_table = pd.DataFrame(
    [{k: item[k] for k in ("model", "accuracy", "precision", "recall", "f1", "auc")} for item in metrics]
)
regression_table = pd.DataFrame([regression])[["mae", "rmse", "r2", "adjusted_r2"]]
print("Classifier metrics")
display(classifier_table)
print("Regression metrics (separate scale)")
display(regression_table)

best_name = max(metrics, key=lambda item: (item["f1"], item["auc"]))["model"]
best_pipeline = fitted[best_name]
joblib.dump(best_pipeline, PIPELINE_PATH)
reloaded = joblib.load(PIPELINE_PATH)
sample = x_test.head(1)
original = float(best_pipeline.predict(sample)[0])
reloaded_pred = float(reloaded.predict(sample)[0])
print("deploy_candidate", best_name)
print("reload_check", {"original": original, "reloaded": reloaded_pred, "match": original == reloaded_pred})

report = {
    "class_balance": balance,
    "train_size": int(len(x_train)),
    "test_size": int(len(x_test)),
    "train_survived_share": round(float(y_train.mean()), 4),
    "test_survived_share": round(float(y_test.mean()), 4),
    "preprocessing": {
        "numeric": "median impute + standard scale, fit on train only",
        "categorical": "most-frequent impute + one-hot, fit on train only",
        "why_it_differs_from_eda": (
            "EDA drops rare missing rows and the deck column for the story. "
            "The saved model must score raw rows, so imputation stays inside "
            "the pipeline instead of deleting rows before the split."
        ),
    },
    "classifiers": [
        {k: item[k] for k in ("model", "confusion_matrix", "accuracy", "precision", "recall", "f1", "auc")}
        for item in metrics
    ],
    "imbalance": imbalance,
    "random_forest_search": tuning,
    "regression": regression,
    "deploy_candidate": best_name,
    "reload_check": {
        "original_prediction": original,
        "reloaded_prediction": reloaded_pred,
        "match": original == reloaded_pred,
    },
}
REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(f"Wrote {REPORT_PATH.name}")



**Recommendation.** Deploy the **decision tree**. It ties random forest for accuracy and has the best F1 and recall of the three untuned classifiers. Logistic regression has a higher AUC but misses more survivors. `survival_pipeline.joblib` stores the full preprocessing + estimator pipeline so raw rows can be scored end to end.

